In [225]:
import warnings
warnings.filterwarnings('ignore')

In [226]:
import pandas as pd
import numpy as np
import pyreadr
import joblib
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import plotly.express as px

from tqdm import tqdm

from dash import Dash, html, dcc, dash_table, Input, Output, callback
import dash_bootstrap_components as dbc

In [227]:
pd.set_option('display.max_columns', 100)

In [228]:
dem_uncont = 34
rep_uncont = 31

In [229]:
seat_sims = pyreadr.read_r('model_output/tot_seats_sims.RDS')[None]
seat_sims = seat_sims.rename({None: 'seats'}, axis=1)
seat_sims['seats'] = seat_sims['seats'].map(lambda x: x + dem_uncont)
seat_sims['winner'] = seat_sims['seats'].map(lambda x: 'Democrats' if x >= 51 else 'Republicans')
seat_sims

,seats,winner
0,54,Democrats
1,55,Democrats
2,50,Republicans
3,53,Democrats
4,48,Republicans
...,...,...
7995,51,Democrats
7996,51,Democrats
7997,53,Democrats
7998,52,Democrats


In [230]:
sim_counts = seat_sims.groupby(['seats']).count().reset_index().rename({'winner': 'count'}, axis=1)
n_sims = seat_sims.shape[0]
sim_counts['pct'] = sim_counts['count'] / n_sims * 100
sim_counts['winner'] = sim_counts['seats'].map(lambda x: 'Democrats' if x >= 51 else 'Republicans')
seat_sims = pd.merge(left=seat_sims.drop(['winner'], axis=1), right=sim_counts, on='seats', how='left')
def get_desc(winner, pct, seats):
    return f'{winner} wins {seats if seats >= 51 else (100-seats)} seats in {pct:.2f}% of simulations'
#seat_sims['desc'] = seat_sims[['winner', 'pct', 'seats']].apply(lambda x: get_desc(x['winner'], x['pct'], x['seats']), axis=1)
sim_counts.head()

,seats,count,pct,winner
0,36,1,0.0125,Republicans
1,38,1,0.0125,Republicans
2,40,2,0.0250,Republicans
3,41,3,0.0375,Republicans
4,42,12,0.1500,Republicans


In [231]:
np.unique(seat_sims['seats']).shape[0]

27

In [232]:
sims_hist = px.histogram(seat_sims, x='seats', nbins=np.unique(seat_sims['seats']).shape[0]*2, color='winner',
                         color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},
                         labels={'seats':'Seats won by Democrats', 'winner': 'Winner'}, template='plotly_white')
sims_hist.update_traces(showlegend=False)
sims_hist.add_vline(x=50.5, line_width=1, line_color='black', annotation_text='51 seats required for Dem majority', 
                    annotation_position='top right')
sims_hist

In [233]:
output_ts = pd.read_csv('model_output/output_over_time.csv')
output_ts.head()

,date,y,geo,type
0,2026-09-27,51.348375,US Senate,seats
1,2026-09-27,58.600000,US Senate,chance
2,2026-09-27,3.816193,US Senate,seats_sd
3,2026-09-27,38.771876,AL,y_pred
4,2026-09-27,3.537242,AL,y_pred_sd


In [234]:
chance_over_time = output_ts[(output_ts['geo'] == 'US Senate') &
                             (output_ts['type'] == 'chance')]
chance_over_time['rep_chance'] = chance_over_time['y'].map(lambda x: 100 - x)
chance_over_time = chance_over_time.rename({'y': 'Democrats', 'rep_chance': 'Republicans'}, axis=1)
chance_time_ser = px.line(chance_over_time, x='date', y=['Democrats', 'Republicans'], template='plotly_white',
                         color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},)
chance_time_ser.update_traces(hovertemplate="%{y:.1f}%")
chance_time_ser.update_layout(
    xaxis=dict(range=[pd.to_datetime('2026-09-27'), pd.to_datetime('2026-11-10')]),
    yaxis=dict(range=[0, 100]),
    xaxis_title='Date',
    yaxis_title='Win Probability (%)',
    title=dict(text="Senate Win Probability Over Time"),
    hovermode="x",
    showlegend=False
)
chance_time_ser

In [235]:
seats_over_time = output_ts[(output_ts['geo'] == 'US Senate') &
                             (output_ts['type'] == 'seats')]
seats_over_time['rep_seats'] = seats_over_time['y'].map(lambda x: 100 - x)
seats_over_time = seats_over_time.rename({'y': 'Democrats', 'rep_seats': 'Republicans'}, axis=1)
seats_time_ser = px.line(seats_over_time, x='date', y=['Democrats', 'Republicans'], template='plotly_white',
                        color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},)
seats_time_ser.update_traces(hovertemplate="%{y:.1f}")
seats_time_ser.update_layout(
    xaxis=dict(range=[pd.to_datetime('2026-09-27'), pd.to_datetime('2026-11-10')]),
    yaxis=dict(range=[0, 100]),
    xaxis_title='Date',
    yaxis_title='Average Seats Over All Simulations',
    title=dict(text="Projected Seats Over Time"),
    hovermode="x",
    showlegend=False
)
seats_time_ser

In [236]:
joblib.dump(sims_hist, 'display_data/sims_histogram.pkl')
joblib.dump(chance_time_ser, 'display_data/chance_time_ser.pkl')
joblib.dump(seats_time_ser, 'display_data/seats_time_ser.pkl')

['display_data/seats_time_ser.pkl']

In [237]:
# posterior prediction
post = pyreadr.read_r('model_output/labeled_posterior.RDS')[None]
post_untransp = post.copy()

In [238]:
post = post.T
post.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,...,7950,7951,7952,7953,7954,7955,7956,7957,7958,7959,7960,7961,7962,7963,7964,7965,7966,7967,7968,7969,7970,7971,7972,7973,7974,7975,7976,7977,7978,7979,7980,7981,7982,7983,7984,7985,7986,7987,7988,7989,7990,7991,7992,7993,7994,7995,7996,7997,7998,7999
AL,-5.240027,-11.773363,-9.716692,-9.985318,-7.795254,-11.276411,-9.778336,-3.544404,-10.037415,-15.472867,1.026506,-11.074172,-8.854900,-9.505752,-10.209108,-7.600732,-7.818334,-10.354227,-7.945273,-5.406140,-10.684047,-7.023360,-5.014891,-7.245841,-6.727597,-12.586274,-12.040464,-5.993769,-11.044674,-11.210220,-8.599710,-9.778827,-9.491266,-6.023136,-14.112218,-13.093121,-5.893678,-8.932703,-6.617532,-10.686520,-8.572274,-6.806890,-11.893740,-8.742760,-10.893063,-11.451583,-10.955686,-13.019154,-5.973411,-9.515903,...,-9.679023,-9.037057,-7.844016,-13.127281,-5.089271,-9.750433,-10.856206,-5.739663,-10.212484,-6.250848,-14.258799,-14.260881,-8.520904,-5.195106,-15.542828,-5.135859,-14.494218,-1.046483,-9.978126,-6.883747,-9.102517,-7.776520,-2.692190,-19.305942,-1.339206,-11.885447,-9.523043,-11.540368,-5.811572,-10.535950,-8.786247,-10.823570,-8.940196,-6.014203,-14.762712,-10.345603,-8.965859,-16.790123,-7.969877,-8.071331,-8.935740,-12.160005,-8.625608,-8.511844,-10.170376,-11.035231,-13.039599,-7.913876,-10.696450,-10.207194
AK,0.716138,5.825318,0.053799,5.737606,-0.899330,4.784589,5.067249,5.329638,-0.422111,-5.499269,11.058017,0.472230,-0.846200,6.713180,-0.591953,3.244215,5.808578,4.851647,-0.221759,3.004032,7.073469,4.329273,2.502608,5.281264,5.117252,1.507612,2.528662,3.803973,-0.781385,1.655091,3.543950,5.896971,7.180305,8.449687,2.948698,-0.147757,3.813720,0.165858,-0.883570,3.402552,-1.142543,3.857666,7.193277,4.185284,2.784325,-1.597565,1.104606,3.211189,2.291274,4.586268,...,4.080850,3.010115,10.179278,-0.126135,7.721823,2.796021,1.440068,3.766834,-1.330938,10.659182,2.545774,-0.762684,7.179127,3.468418,3.888504,1.727653,4.927890,2.349653,-0.423091,7.469002,1.915206,3.547743,8.421634,0.344320,4.012690,-2.106007,6.640175,4.976614,3.937018,0.953066,-2.995725,5.307869,-0.546141,6.130668,1.820884,3.196047,3.414483,7.314958,4.225047,4.163687,0.141844,4.362153,4.037151,0.399401,4.054498,1.132571,1.855440,0.806322,3.666773,1.862839
AR,-10.715384,-12.609767,-8.985176,-11.715151,-10.424204,-8.603952,-9.710469,-6.247300,-14.670470,-14.720768,-1.671601,-12.322857,-13.220908,-7.846104,-12.701792,-11.262277,-10.612981,-9.085538,-11.381766,-7.807504,-11.837111,-11.364156,-7.585973,-7.834979,-12.072980,-9.333102,-11.821155,-8.506061,-11.804251,-11.618367,-8.984240,-9.679684,-7.173804,-7.958617,-13.994326,-13.409496,-6.843190,-10.238171,-11.795176,-10.873023,-10.842371,-6.872409,-12.802532,-14.433859,-13.141821,-11.363838,-9.552783,-14.504235,-11.202054,-12.473581,...,-13.422313,-11.835086,-8.512667,-14.081877,-4.297124,-12.762186,-13.365942,-12.032353,-10.749862,-4.612436,-15.110425,-12.674835,-8.707960,-8.153221,-14.009460,-5.235687,-16.017327,-1.759726,-9.812463,-8.963509,-12.256618,-6.855491,-5.251235,-16.070281,-3.744054,-17.043686,-9.770889,-16.519747,-5.846989,-10.227454,-9.676044,-11.299422,-10.210265,-3.109793,-17.915163,-10.241002,-8.296131,-12.929235,-6.475032,-12.721776,-11.633140,-11.196764,-9.232351,-7.480202,-11.658561,-14.078325,-17.622161,-11.820813,-8.077963,-11.266972
CO,11.269828,12.055284,7.130550,13.977760,8.731036,12.425205,14.217022,13.115943,10.682637,7.011339,13.173072,13.665758,10.780587,16.039849,7.795757,15.525007,13.576654,11.668282,6.676158,5.946164,12.801129,18.565703,9.769016,16.299779,13.552020,11.349200,11.176026,19.924556,7.593673,13.396010,13.954451,10.336346,9.254865,10.348469,11.640186,5.681400,16.189909,11.071193,15.294789,9.074571,12.929454,11.901672,11.534357,13.466858,10.462826,11.599261,7.521361,10.985021,12.264513,9.403665,...,13.685568,8.506568,17.283224,10.115102,13.090742,6.784750,11.520730,1

In [239]:
sim_corr = post_untransp.corr()

In [240]:
sim_corr.head()

,AL,AK,AR,CO,DE,FL,GA,ID,IL,IA,KS,KY,LA,ME,MA,MI,MN,MS,MT,NE,NH,NJ,NM,NC,OH,OK,OR,RI,SC,SD,TN,TX,VA,WV,WY
AL,1.000000,0.516202,0.738736,0.490234,0.497080,0.698951,0.529667,0.735457,0.704534,0.739173,0.731714,0.540759,0.749397,0.731217,0.519232,0.728889,0.704711,0.742867,0.693596,0.720809,0.498497,0.506464,0.505063,0.702685,0.503813,0.756767,0.506986,0.487379,0.488054,0.512253,0.556170,0.693590,0.504822,0.739232,0.737287
AK,0.516202,1.000000,0.518569,0.521586,0.525931,0.507992,0.557892,0.526914,0.504517,0.522385,0.521818,0.568620,0.525990,0.523610,0.560711,0.513415,0.498545,0.518620,0.495327,0.517843,0.561703,0.539983,0.554023,0.498904,0.554639,0.522686,0.547995,0.549659,0.539330,0.543659,0.598431,0.507287,0.544419,0.521400,0.520710
AR,0.738736,0.518569,1.000000,0.487281,0.499840,0.703763,0.526897,0.737928,0.699201,0.740979,0.736522,0.537503,0.742054,0.732236,0.522411,0.728377,0.698976,0.726027,0.693419,0.720786,0.508488,0.500900,0.501599,0.704451,0.504920,0.744104,0.501885,0.494645,0.495292,0.514548,0.556798,0.698749,0.499136,0.735672,0.734963
CO,0.490234,0.521586,0.487281,1.000000,0.708934,0.651471,0.550592,0.497650,0.653935,0.502188,0.485067,0.549363,0.503908,0.484474,0.537293,0.486686,0.642071,0.493575,0.652283,0.475081,0.709519,0.715785,0.719738,0.646764,0.703160,0.499774,0.710417,0.700593,0.697709,0.528054,0.571412,0.653878,0.703958,0.496893,0.494080
DE,0.497080,0.525931,0.499840,0.708934,1.000000,0.665525,0.543256,0.504462,0.664569,0.507100,0.498051,0.547882,0.503229,0.501632,0.548384,0.501193,0.663530,0.499342,0.662863,0.487620,0.718354,0.725911,0.728646,0.660993,0.714065,0.515780,0.709994,0.711585,0.707692,0.519399,0.577980,0.657619,0.719023,0.499510,0.509410


In [241]:
post.shape

(35, 8000)

In [242]:
def get_tipping_point(sim):
    """
    :param sim: Series representing one posterior draw or "simulation"
    :type sim: pd.Series
    """
    seats_won_dem = np.sum(sim > 0)
    if seats_won_dem >= 51 - dem_uncont: # Democrats win House in this draw
        sim = sim.sort_values(ascending=True)
        won_seats = sim[sim > 0]
        seat_margin = seats_won_dem - (51 - dem_uncont)
    else: # Republicans win House in this draw
        sim = sim.sort_values(ascending=False)
        won_seats = sim[sim < 0]
        seat_margin = (100 - seats_won_dem - dem_uncont - rep_uncont) - (50 - rep_uncont)
    return won_seats.iloc[seat_margin - 1], won_seats.index[seat_margin - 1]

In [243]:
tipping_points = np.array([]) # tipping point for *each sim*

for i in tqdm(range(post.shape[1])):
    sim = post.iloc[:, i]
    _, tp_seat = get_tipping_point(sim)
    tipping_points = np.append(tipping_points, tp_seat)

tipping_points

100%|████████████████████████████████████████████████████████████████████████████| 8000/8000 [00:06<00:00, 1218.17it/s]


array(['NE', 'ME', 'WY', ..., 'AK', 'IA', 'AK'],
      shape=(8000,), dtype='<U32')

In [244]:
data = pd.read_csv('../../model_output/senate_predictions.csv')
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.464026,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.000000,0.0,0.000000,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.162416,-48.013273,0,-96.026546,40.999159,3.547210,0.5750,1,34.148394,48.120165
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.464026,0,1,50.113259,3.855569,48.010482,3.855569,Pacific,2026,0.000000,0.0,0.000000,5,Senate,7252.414198,3.855569,1.963560,-2.102777,-4.446579,35.161107,-1,70.322215,53.306305,3.400788,83.4000,2,46.666778,60.101721
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.464026,0,1,39.806386,1.161279,52.528412,1.161279,West South Central,2026,0.000000,0.0,0.000000,5,Senate,2746.326849,1.161279,1.077627,12.722026,-22.157017,2.405409,-1,4.810817,39.657849,3.571207,0.1875,3,32.532863,46.605455
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",6665758.10,134935.30,6800693.40,98.015860,1.984140,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-8.464026,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9607.108748,0.000000,0.000000,0.000000,20.393162,48.015860,1,96.031719,61.626401,3.503947,99.9500,4,54.839861,68.526059
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",2206861.15,60855.60,2267716.75,97.316437,2.683563,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-8.464026,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9470.488976,0.000000,0.000000,0.000000,24.486415,47.316437,1,94.632875,63.102567,3.504368,99.9875,5,56.265119,70.054506


In [245]:
data['tipping_point_prob'] = data['state_po'].map(lambda x: np.mean(tipping_points == x) * 100)
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.464026,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.000000,0.0,0.000000,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.162416,-48.013273,0,-96.026546,40.999159,3.547210,0.5750,1,34.148394,48.120165,0.0000
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.464026,0,1,50.113259,3.855569,48.010482,3.855569,Pacific,2026,0.000000,0.0,0.000000,5,Senate,7252.414198,3.855569,1.963560,-2.102777,-4.446579,35.161107,-1,70.322215,53.306305,3.400788,83.4000,2,46.666778,60.101721,5.5500
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.464026,0,1,39.806386,1.161279,52.528412,1.161279,West South Central,2026,0.000000,0.0,0.000000,5,Senate,2746.326849,1.161279,1.077627,12.722026,-22.157017,2.405409,-1,4.810817,39.657849,3.571207,0.1875,3,32.532863,46.605455,0.0000
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",6665758.10,134935.30,6800693.40,98.015860,1.984140,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-8.464026,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9607.108748,0.000000,0.000000,0.000000,20.393162,48.015860,1,96.031719,61.626401,3.503947,99.9500,4,54.839861,68.526059,0.0625
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",2206861.15,60855.60,2267716.75,97.316437,2.683563,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-8.464026,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9470.488976,0.000000,0.000000,0.000000,24.486415,47.316437,1,94.632875,63.102567,3.504368,99.9875,5,56.265119,70.054506,0.2250


In [246]:
data.sort_values('tipping_point_prob', ascending=False).head(5)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
15,15,Michigan,Abdul El-Sayed,Mike J. Rogers,False,False,D,MI,"EL-SAYED, ABDUL","ROGERS, MICHAEL J",4.719966e+07,15432313.28,6.263198e+07,75.360329,24.639671,-0.192456,49.278763,77.709102,2.333008,4.148895,12.968600,0.319413,32.445883,7404258,2673073,10077331,73.474395,26.525605,0.987401,-8.464026,0,0,49.401904,11.809941,45.486544,11.809941,East North Central,2026,0.0,0.00000,0.00000,2,Senate,5679.179195,11.809941,3.436559,-3.915360,8.079114,25.360329,0,50.720658,50.764505,3.540839,58.4125,16,43.981085,57.774942,10.8250
13,13,Maine,Troy Jackson,Susan Collins,False,True,R,ME,"JACKSON, TROY DALE","COLLINS, SUSAN M.",2.397712e+07,15299978.61,3.927710e+07,61.046056,38.953944,3.820912,53.544306,94.249770,0.897865,1.689465,0.930085,0.342309,36.189622,526309,836050,1362359,38.632181,61.367819,0.987401,-8.464026,0,1,51.137936,9.264539,46.002049,9.264539,New England,2026,0.0,0.00000,0.00000,3,Senate,3726.620896,9.264539,3.043771,-5.135888,16.105850,11.046056,-1,22.092111,50.642718,3.583450,56.8125,14,43.599989,57.623727,10.3625
31,31,Texas,James Talarico,Ken Paxton,False,False,R,TX,"TALARICO, JAMES","PAXTON, WARREN KENNETH JR.",1.397766e+08,20997954.85,1.607746e+08,86.939504,13.060496,-5.916874,43.060966,48.286718,4.530331,31.642337,13.173480,0.180615,33.823334,24400697,4744808,29145505,83.720275,16.279725,0.987401,-8.464026,0,0,50.163685,14.805073,46.239477,14.805073,West South Central,2026,0.0,0.25284,-0.25284,6,Senate,7558.477337,14.805073,3.847736,-3.924208,-3.369722,36.939504,0,73.879008,52.095501,3.651079,71.7625,32,44.953149,59.354262,9.5625
10,10,Kansas,Adam Hamilton,Roger Marshall,False,True,R,KS,"HAMILTON, ADAM","MARSHALL, ROGER W",1.652601e+07,2514153.97,1.904016e+07,86.795522,13.204478,-8.032578,41.794263,80.203805,2.235100,8.765322,5.263831,0.443291,35.603269,2124059,813821,2937880,72.299039,27.700961,0.987401,-8.464026,0,1,49.320444,5.817582,47.131473,5.817582,West North Central,2026,0.0,0.00000,0.00000,2,Senate,7533.462650,5.817582,2.411966,-2.188971,-7.601130,36.795522,-1,73.591044,49.709064,3.537171,46.4250,11,42.790235,56.748509,8.6375
24,24,Ohio,Sherrod Brown,Jon Husted,False,True,R,OH,"BROWN, SHERROD","HUSTED, JON",5.786194e+07,9843192.51,6.770513e+07,85.461675,14.538325,-5.266554,44.343385,80.925356,1.708882,3.291326,11.486903,0.076606,31.487184,9001099,2798349,11799448,76.284069,23.715931,0.987401,-8.464026,0,1,48.026016,5.933895,45.544410,5.933895,East North Central,2026,0.0,0.00000,0.00000,2,Senate,7303.697897,5.933895,2.435959,-2.481606,-2.069082,35.461675,-1,70.923350,50.886922,3.476115,60.0875,25,44.003236,57.769375,8.5000


In [247]:
def get_rating(dem_chance):
    if dem_chance > 100:
        raise ValueError('Invalid win chance.')
    if dem_chance > 95:
        return 'Safe D'
    elif dem_chance >= 90:
        return 'Very Likely D'
    elif dem_chance >= 75:
        return 'Likely D'
    elif dem_chance >= 65:
        return 'Lean D'
    elif dem_chance >= 60:
        return 'Tilt D'
    elif dem_chance >= 40:
        return 'Tossup'
    elif dem_chance >= 35:
        return 'Tilt R'
    elif dem_chance >= 25:
        return 'Lean R'
    elif dem_chance >= 10:
        return 'Likely R'
    elif dem_chance >= 5:
        return 'Very Likely R'
    else:
        return 'Safe R'

def get_matchup(dem_cand, rep_cand, dem_inc_any, rep_inc_any):
    indie_d = ['Dan Osborn', 'Brian Bengs', 'Todd Achilles', 'Seth Bodnar']
    indie_r = []
    
    if dem_cand in indie_d:
        dem_lab = dem_cand + f'{'*' if dem_inc_any == True else ''}' + ' (I)'
        dem_color = '#792ba6'
    else:
        dem_lab = dem_cand + f'{'*' if dem_inc_any == True else ''}' + ' (D)'
        dem_color = '#366bbf'
    
    if rep_cand in indie_r:
        rep_lab = rep_cand + f'{'*' if rep_inc_any == True else ''}' + ' (I)'
        rep_color = '#792ba6'
    else:
        rep_lab = rep_cand + f'{'*' if rep_inc_any == True else ''}' + ' (R)'
        rep_color = '#e63929'

    return f'<b style="color:{dem_color};">' + dem_lab + f'</b> vs <b style="color:{rep_color};">' + rep_lab + '</b>'

In [248]:
data['rating'] = data['chance'].map(lambda x: get_rating(x))
for party in ['dem', 'rep']:
    data[f'{party}_cand'] = data[f'{party}_cand'].map(lambda x: 'TBD' if x[:3] == 'TBD' else x)
data['matchup'] = data[['dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any']].apply(lambda x: get_matchup(x['dem_cand'], x['rep_cand'],
                                                                                                          x['dem_inc_any'], x['rep_inc_any']), axis=1)
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.464026,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.000000,0.0,0.000000,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.162416,-48.013273,0,-96.026546,40.999159,3.547210,0.5750,1,34.148394,48.120165,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b..."
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.464026,0,1,50.113259,3.855569,48.010482,3.855569,Pacific,2026,0.000000,0.0,0.000000,5,Senate,7252.414198,3.855569,1.963560,-2.102777,-4.446579,35.161107,-1,70.322215,53.306305,3.400788,83.4000,2,46.666778,60.101721,5.5500,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>..."
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.464026,0,1,39.806386,1.161279,52.528412,1.161279,West South Central,2026,0.000000,0.0,0.000000,5,Senate,2746.326849,1.161279,1.077627,12.722026,-22.157017,2.405409,-1,4.810817,39.657849,3.571207,0.1875,3,32.532863,46.605455,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<..."
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",6665758.10,134935.30,6800693.40,98.015860,1.984140,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-8.464026,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9607.108748,0.000000,0.000000,0.000000,20.393162,48.015860,1,96.031719,61.626401,3.503947,99.9500,4,54.839861,68.526059,0.0625,Safe D,"<b style=""color:#366bbf;"">John Hickenlooper* (..."
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",2206861.15,60855.60,2267716.75,97.316437,2.683563,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-8.464026,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9470.488976,0.000000,0.000000,0.000000,24.486415,47.316437,1,94.632875,63.102567,3.504368,99.9875,5,56.265119,70.054506,0.2250,Safe D,"<b style=""color:#366bbf;"">Chris Coons* (D)</b>..."


In [249]:
data['projected_winner'] = data['chance'].map(lambda x: 'D' if x > 50 else 'R')
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.464026,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.162416,-48.013273,0,-96.026546,40.999159,3.547210,0.5750,1,34.148394,48.120165,0.00,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.464026,0,1,50.113259,3.855569,48.010482,3.855569,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.855569,1.963560,-2.102777,-4.446579,35.161107,-1,70.322215,53.306305,3.400788,83.4000,2,46.666778,60.101721,5.55,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.464026,0,1,39.806386,1.161279,52.528412,1.161279,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.161279,1.077627,12.722026,-22.157017,2.405409,-1,4.810817,39.657849,3.571207,0.1875,3,32.532863,46.605455,0.00,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R


In [250]:
data['hold'] = data['projected_winner'] ==  data['current_party']
data['flip'] = data['hold'].map(lambda x: not x)
data['flip_indic'] = data['flip'].map(lambda x: 'Flip' if x else '')
#data['flip'] = data['flip'].map(lambda x: 'Yes' if x else 'No')
data.head(2)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.464026,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.819209,1.34878,10.898934,-21.162416,-48.013273,0,-96.026546,40.999159,3.547210,0.575,1,34.148394,48.120165,0.00,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.464026,0,1,50.113259,3.855569,48.010482,3.855569,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.855569,1.96356,-2.102777,-4.446579,35.161107,-1,70.322215,53.306305,3.400788,83.400,2,46.666778,60.101721,5.55,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip


In [251]:
data['projected_2p_margin'] = data['y_pred'].map(lambda y_pred: f'D+{y_pred - (100-y_pred):.1f}' if y_pred > 50 else f'R+{(100-y_pred) - y_pred:.1f}')
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.464026,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.162416,-48.013273,0,-96.026546,40.999159,3.547210,0.5750,1,34.148394,48.120165,0.00,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+18.0
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.464026,0,1,50.113259,3.855569,48.010482,3.855569,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.855569,1.963560,-2.102777,-4.446579,35.161107,-1,70.322215,53.306305,3.400788,83.4000,2,46.666778,60.101721,5.55,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.6
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.464026,0,1,39.806386,1.161279,52.528412,1.161279,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.161279,1.077627,12.722026,-22.157017,2.405409,-1,4.810817,39.657849,3.571207,0.1875,3,32.532863,46.605455,0.00,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+20.7


In [252]:
data['rep_chance'] = data['chance'].map(lambda x: 100 - x)
data['rounded_dem_chance'] = data['chance'].map(lambda x: np.round(x, 1))
data['rounded_rep_chance'] = data['rep_chance'].map(lambda x: np.round(x, 1))
data['disp_dem_chance'] = data['rounded_dem_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data['disp_rep_chance'] = data['rounded_rep_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.464026,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.162416,-48.013273,0,-96.026546,40.999159,3.547210,0.5750,1,34.148394,48.120165,0.00,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+18.0,99.4250,0.6,99.4,<1%,>99%
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.464026,0,1,50.113259,3.855569,48.010482,3.855569,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.855569,1.963560,-2.102777,-4.446579,35.161107,-1,70.322215,53.306305,3.400788,83.4000,2,46.666778,60.101721,5.55,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.6,16.6000,83.4,16.6,83.4%,16.6%
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.464026,0,1,39.806386,1.161279,52.528412,1.161279,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.161279,1.077627,12.722026,-22.157017,2.405409,-1,4.810817,39.657849,3.571207,0.1875,3,32.532863,46.605455,0.00,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+20.7,99.8125,0.2,99.8,<1%,>99%


In [253]:
data['swing_24_to_26'] = data['y_pred'].astype(float).map(lambda x: x - (100 - x)) - data['dem_2p_24'].astype(float).map(lambda x: x - (100 - x))
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance,swing_24_to_26
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.464026,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.162416,-48.013273,0,-96.026546,40.999159,3.547210,0.5750,1,34.148394,48.120165,0.00,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+18.0,99.4250,0.6,99.4,<1%,>99%,12.879541
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.464026,0,1,50.113259,3.855569,48.010482,3.855569,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.855569,1.963560,-2.102777,-4.446579,35.161107,-1,70.322215,53.306305,3.400788,83.4000,2,46.666778,60.101721,5.55,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.6,16.6000,83.4,16.6,83.4%,16.6%,20.305734
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.464026,0,1,39.806386,1.161279,52.528412,1.161279,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.161279,1.077627,12.722026,-22.157017,2.405409,-1,4.810817,39.657849,3.571207,0.1875,3,32.532863,46.605455,0.00,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+20.7,99.8125,0.2,99.8,<1%,>99%,10.655685


In [254]:
data['disp_24_to_26_swing'] = data['swing_24_to_26'].map(lambda x: f'D+{x:.1f}' if x > 0 else f'R+{abs(x):.1f}')
data.head(2)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance,swing_24_to_26,disp_24_to_26_swing
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.464026,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.819209,1.34878,10.898934,-21.162416,-48.013273,0,-96.026546,40.999159,3.547210,0.575,1,34.148394,48.120165,0.00,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+18.0,99.425,0.6,99.4,<1%,>99%,12.879541,D+12.9
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.464026,0,1,50.113259,3.855569,48.010482,3.855569,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.855569,1.96356,-2.102777,-4.446579,35.161107,-1,70.322215,53.306305,3.400788,83.400,2,46.666778,60.101721,5.55,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.6,16.600,83.4,16.6,83.4%,16.6%,20.305734,D+20.3


In [255]:
tab_data = data[['state', 'dem_cand', 'rep_cand', 'rating', 'disp_dem_chance', 'disp_rep_chance', 'projected_2p_margin', 'disp_24_to_26_swing',
                 'tipping_point_prob']]
for party in ['dem', 'rep']:
    tab_data[f'disp_{party}_chance'] = tab_data[f'disp_{party}_chance'].map(lambda x: f'<p style="color:{'blue' if party == 'dem' else 'red'};">{x}</p>')
    tab_data[f'{party}_cand'] = tab_data[f'{party}_cand'].map(lambda x: f'{x} (Ind)' if x in ['Dan Osborn', 'Brian Bengs', 
                                                                                              'Todd Achilles', 'Seth Bodnar'] else x)
tab_data = tab_data.rename({
    'state': 'State',
    'dem_cand': 'Democrat',
    'rep_cand': 'Republican',
    'rating': 'Rating',
    'disp_dem_chance': 'Dem Chance',
    'disp_rep_chance': 'Rep Chance',
    'projected_2p_margin': 'Projected Margin',
    'tipping_point_prob': 'Tipping Point Chance',
    'disp_24_to_26_swing': 'Swing from 2024 Pres'
}, axis=1)
tab_data.head()

,State,Democrat,Republican,Rating,Dem Chance,Rep Chance,Projected Margin,Swing from 2024 Pres,Tipping Point Chance
0,Alabama,Everett Weiss,Barry Moore,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+18.0,D+12.9,0.0000
1,Alaska,Mary Peltola,Dan S. Sullivan,Likely D,"<p style=""color:blue;"">83.4%</p>","<p style=""color:red;"">16.6%</p>",D+6.6,D+20.3,5.5500
2,Arkansas,Hallie Shoffner,Tom Cotton,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+20.7,D+10.7,0.0000
3,Colorado,John Hickenlooper,Mark Baisley,Safe D,"<p style=""color:blue;"">>99%</p>","<p style=""color:red;""><1%</p>",D+23.3,D+12.0,0.0625
4,Delaware,Chris Coons,Michael Katz,Safe D,"<p style=""color:blue;"">>99%</p>","<p style=""color:red;""><1%</p>",D+26.2,D+11.2,0.2250


In [256]:
data['rep_pred'] = data['y_pred'].map(lambda x: 100 - x)
data['projected_2p_margin_number'] = data['rep_pred'] - data['y_pred']
data['proj_seat_lean'] = data['projected_2p_margin_number'] - data['generic_ballot_avg']
sv_bias = float(data.sort_values('tipping_point_prob', ascending=False).reset_index().loc[0, 'proj_seat_lean']) # Seats-votes bias, + = R, - = D
sv_bias

6.935014991412682

In [257]:
topline_stats = pd.read_csv('display_data/topline_stats.csv')
if 'sv_bias' in topline_stats['vars'].values:
    topline_stats = topline_stats[topline_stats['vars'] != 'sv_bias']
topline_stats = pd.concat([topline_stats, pd.DataFrame({'vars': ['sv_bias'], 'x': sv_bias})], axis=0)
topline_stats.to_csv('display_data/topline_stats.csv')
topline_stats

,vars,x
0,means_seats_tot,52.173125
1,chamber_win_chance,65.000000
0,sv_bias,6.935015


In [258]:
mean_seats_tot = topline_stats[topline_stats['vars'] == 'means_seats_tot']['x'].values[0]
chamber_win_chance = topline_stats[topline_stats['vars'] == 'chamber_win_chance']['x'].values[0]

In [259]:
chances = topline_stats[topline_stats['vars'] == 'chamber_win_chance'].set_index(['vars']).T
chances['Republicans'] = chances['chamber_win_chance'].map(lambda x: 100 - x)
chances = chances.rename({'chamber_win_chance': 'Democrats'}, axis=1).T.reset_index()
chances = chances.rename({'x': 'Win Probability'}, axis=1)
seats = topline_stats[topline_stats['vars'] == 'means_seats_tot'].set_index(['vars']).T
seats['Republicans'] = seats['means_seats_tot'].map(lambda x: 100 - x)
seats = seats.rename({'means_seats_tot': 'Democrats'}, axis=1).T.reset_index().rename({'x': 'Seat Share'}, axis=1)
summary_stats = pd.merge(left=chances, right=seats, on='vars', how='inner').rename({'vars': 'Party'}, axis=1)
summary_stats.to_csv('display_data/summary_stats.csv')
summary_stats

,Party,Win Probability,Seat Share
0,Democrats,65.0,52.173125
1,Republicans,35.0,47.826875


In [260]:
incl_cols = ['state', 'dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any', 'rating', 'chance', 'disp_dem_chance',
            'disp_rep_chance', 'projected_2p_margin', 'y_pred', 'flip_indic', 'matchup', 'swing_24_to_26', 'disp_24_to_26_swing']
disp_data = data[incl_cols]

In [261]:
rating_ts = output_ts[output_ts['type'] == 'chance']
rating_ts['rating'] = rating_ts['y'].map(get_rating)
rating_ts['date'] = pd.to_datetime(rating_ts['date'])
rating_ts.head()

,date,y,geo,type,rating
1,2026-09-27,58.6000,US Senate,chance,Tossup
5,2026-09-27,0.0500,AL,chance,Safe R
10,2026-09-27,79.8875,AK,chance,Likely D
15,2026-09-27,0.1250,AR,chance,Safe R
20,2026-09-27,99.9625,CO,chance,Safe D


In [262]:
## Alert when there's a rating change
today_date = rating_ts['date'].max()
yesterday = rating_ts[rating_ts['date'] != today_date]['date'].max()

today_ratings = rating_ts[rating_ts['date'] == today_date]
yester_ratings = rating_ts[rating_ts['date'] == yesterday]
compare_df = pd.merge(left=today_ratings, right=yester_ratings, on='geo', how='left', suffixes=['_t', '_y'])
compare_df.head()

,date_t,y_t,geo,type_t,rating_t,date_y,y_y,type_y,rating_y
0,2026-11-01,65.0000,US Senate,chance,Lean D,2026-10-31,62.2000,chance,Tilt D
1,2026-11-01,0.5750,AL,chance,Safe R,2026-10-31,0.5750,chance,Safe R
2,2026-11-01,83.4000,AK,chance,Likely D,2026-10-31,82.7625,chance,Likely D
3,2026-11-01,0.1875,AR,chance,Safe R,2026-10-31,0.1750,chance,Safe R
4,2026-11-01,99.9500,CO,chance,Safe D,2026-10-31,99.9500,chance,Safe D


In [263]:
## This is the cell that actually alerts when there's a rating change
if any(compare_df['rating_t'] != compare_df['rating_y']):
    changes_df = compare_df[compare_df['rating_t'] != compare_df['rating_y']][['geo', 'rating_y', 'rating_t']]
    print(changes_df)
else:
    print("No rating changes today!")

          geo rating_y rating_t
0   US Senate   Tilt D   Lean D
25         OH   Tossup   Tilt D
32         TX   Tilt D   Lean D


In [264]:
for i in range(changes_df.shape[0]):
    seat = changes_df.iloc[i, :]['geo']
    rating_y = changes_df.iloc[i, :]['rating_y']
    rating_t = changes_df.iloc[i, :]['rating_t']
    print(f'{seat}: {rating_y} --> {rating_t}')

US Senate: Tilt D --> Lean D
OH: Tossup --> Tilt D
TX: Tilt D --> Lean D


In [265]:
if any(compare_df['rating_t'] != compare_df['rating_y']):
    print(changes_df.shape)

(3, 3)


In [266]:
## Mini-widget: what is the chance of Dems/Reps winning Senate if X state is won by Dems/Reps?
def det_winner(post, state, dem_winner=True):
    '''
    :param post: Posterior simulation matrix.
    :param state: Congressional district.
    :param dem_winner: Whether or not we set Dems or Reps to win `state`. True if Dem win is fixed, False if Rep win is fixed.
    '''
    winners = []
    seats = []
    post_ut = post.T
    if dem_winner:
        post_ut = post_ut[post_ut[state] > 0]
    else:
        post_ut = post_ut[post_ut[state] < 0]
    post_case = post_ut.T
    for i in range(post_case.shape[0]):
        sim = post_case.iloc[:, i]
        dem_seats = np.sum(sim > 0)
        if dem_seats >= 51 - dem_uncont: # Democrats win House in this draw
            winners.append(1)
        else: # Republicans win House in this draw
            winners.append(0)
        seats.append(dem_seats)
    dem_win_chance = np.mean(np.array(winners)) * 100
    dem_mean_seats = np.mean(np.array(seats)) + dem_uncont
    return dem_win_chance, dem_mean_seats

st = 'TX'
dem_winner = True
cond_win_chance, cond_seats = det_winner(post, st, dem_winner)
print(f'If Dems {'win' if dem_winner else 'lose'} {st}-SEN, they have a {cond_win_chance:.1f}% chance of winning the chamber overall, with mean projected seats of {cond_seats:.1f}')

If Dems win TX-SEN, they have a 80.0% chance of winning the chamber overall, with mean projected seats of 53.0


In [267]:
disp_data.to_csv('display_data/choropleth_display_data.csv')
tab_data.to_csv('display_data/table_display_data.csv')
data.to_csv('display_data/all_data.csv')